# LSTM Experiment F All Features

**24-hour input lookback → next-hour EV charging demand.**

Extracted from `Untitled0.ipynb`. Original execution outputs are retained. Run cells in order in Google Colab; confirm Google Drive paths before executing.

## Data loading, preprocessing, model training, evaluation and saved results

## Notebook Setup

In [ ]:

# ============================================================
# EXPERIMENT L
# LSTM - HISTORICAL DEMAND + TRAFFIC + WEATHER + EV SALES
# 24-HOUR LOOKBACK -> NEXT-HOUR EV CHARGING DEMAND
# ============================================================

# ============================================================


## 1. Import Libraries and Mount Google Drive

In [ ]:
# 1. IMPORTS AND GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import json
import random
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dropout,
    Dense
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)


# ============================================================


## 2. Experiment Configuration

In [ ]:
# 2. CONFIGURATION
# ============================================================

ROOT = "/content/drive/MyDrive/EV_Load_Forecasting_FYP"

DATASET_PATH = os.path.join(
    ROOT,
    "Processed Data",
    "Hourly",
    "experiment_c_final_2019.csv"
)

EXP_DIR = os.path.join(
    ROOT,
    "Experiments",
    "LSTM_Experiment_L_All_Features_2019"
)

os.makedirs(EXP_DIR, exist_ok=True)

MODEL_PATH = os.path.join(
    EXP_DIR,
    "lstm_experiment_l_best.keras"
)

LOOKBACK = 24
BATCH_SIZE = 32
EPOCHS = 200
LEARNING_RATE = 0.001

WEATHER_COLS = [
    "clouds_lag1d",
    "dewpt_lag1d",
    "ghi_lag1d",
    "max_uv_lag1d",
    "precip_lag1d",
    "rh_lag1d",
    "slp_lag1d",
    "temp_lag1d",
    "wind_gust_spd_lag1d",
    "wind_spd_lag1d"
]

SALES_COL = "monthly_ev_sales_lag1"
TRAFFIC_COL = "total_traffic"
TARGET_COL = "hourly_energy_kwh"

RAW_FEATURE_COLS = [
    TARGET_COL,
    TRAFFIC_COL,
    SALES_COL
] + WEATHER_COLS

print("Experiment directory:", EXP_DIR)


# ============================================================


## 3. Load and Inspect Dataset

In [ ]:
# 3. LOAD AND VERIFY DATASET
# ============================================================

df = pd.read_csv(
    DATASET_PATH,
    parse_dates=["date_hour"]
)

df = (
    df.sort_values("date_hour")
    .reset_index(drop=True)
)

required_cols = ["date_hour"] + RAW_FEATURE_COLS

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

if missing_cols:
    raise ValueError(
        f"Missing dataset columns: {missing_cols}"
    )

df = df[required_cols].copy()

for col in RAW_FEATURE_COLS:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

print("\n" + "=" * 70)
print("EXPERIMENT L - DATASET INFORMATION")
print("=" * 70)

print("Dataset shape:", df.shape)

print(
    "Date range:",
    df["date_hour"].min(),
    "to",
    df["date_hour"].max()
)

print("\nMissing values:")
print(df.isna().sum())

print("\nFirst five rows:")
print(df.head())

assert len(df) == 8016

assert df[TARGET_COL].isna().sum() == 206

assert not df["date_hour"].duplicated().any()

expected_dates = pd.date_range(
    df["date_hour"].min(),
    df["date_hour"].max(),
    freq="h"
)

assert np.array_equal(
    expected_dates.to_numpy(),
    df["date_hour"].to_numpy()
), "Dataset has missing or irregular hourly timestamps."


# ============================================================


## 4. Chronological Data Splitting

In [ ]:
# 4. CHRONOLOGICAL SPLIT
# ============================================================

n_total = len(df)

train_end = int(n_total * 0.70)
val_end = int(n_total * 0.85)

assert train_end == 5611
assert val_end == 6813

print("\n" + "=" * 70)
print("CHRONOLOGICAL SPLITTING")
print("=" * 70)

for name, part in [
    ("Training", df.iloc[:train_end]),
    ("Validation", df.iloc[train_end:val_end]),
    ("Testing", df.iloc[val_end:])
]:

    print(
        f"\n{name}: {len(part)} rows"
    )

    print(
        "Date range:",
        part["date_hour"].min(),
        "to",
        part["date_hour"].max()
    )

    print(
        "Missing demand:",
        part[TARGET_COL].isna().sum()
    )

    print(
        "Missing traffic:",
        part[TRAFFIC_COL].isna().sum()
    )

    print(
        "Missing sales:",
        part[SALES_COL].isna().sum()
    )


# ============================================================


## 5. Missing Value Handling

In [ ]:
# 5. MISSING VALUE HANDLING
# ============================================================

processed = df.copy()

# Preserve original target values.
# Missing actual targets will not be used for
# training or evaluation.

processed["target_original"] = (
    processed[TARGET_COL]
)

# Historical demand: forward-fill only.
processed["demand_input"] = (
    processed[TARGET_COL].ffill()
)

# Traffic: forward-fill only.
processed["traffic_input"] = (
    processed[TRAFFIC_COL].ffill()
)

# Previous-month EV sales: forward-fill only.
processed["sales_input"] = (
    processed[SALES_COL].ffill()
)

# Weather variables are already lagged by one day.
processed[WEATHER_COLS] = (
    processed[WEATHER_COLS].ffill()
)

# Handle remaining leading missing external values
# using training-only medians.

EXTERNAL_COLS = [
    "traffic_input",
    "sales_input"
] + WEATHER_COLS

imputation_report = []

for col in EXTERNAL_COLS:

    missing_before = int(
        processed[col].isna().sum()
    )

    if missing_before > 0:

        train_median = (
            processed.iloc[:train_end][col].median()
        )

        if pd.isna(train_median):
            raise ValueError(
                f"No valid training observations for {col}"
            )

        processed[col] = (
            processed[col].fillna(train_median)
        )

        imputation_report.append({
            "feature": col,
            "leading_missing_filled": missing_before,
            "training_median": float(train_median)
        })

if processed["demand_input"].isna().any():
    raise ValueError(
        "Historical demand has leading missing values. "
        "Inspect the dataset before continuing."
    )

FEATURE_COLS = [
    "demand_input",
    "traffic_input"
] + WEATHER_COLS + [
    "sales_input"
]

N_FEATURES = len(FEATURE_COLS)

print("\n" + "=" * 70)
print("MISSING VALUE HANDLING")
print("=" * 70)

print(
    processed[
        ["target_original"] + FEATURE_COLS
    ].isna().sum()
)

print("\nInput features:")
for i, col in enumerate(FEATURE_COLS, start=1):
    print(f"{i:02d}. {col}")

print("\nNumber of features:", N_FEATURES)

assert N_FEATURES == 13

assert processed[FEATURE_COLS].notna().all().all()

processed.to_csv(
    os.path.join(
        EXP_DIR,
        "processed_dataset.csv"
    ),
    index=False
)

pd.DataFrame(
    imputation_report,
    columns=[
        "feature",
        "leading_missing_filled",
        "training_median"
    ]
).to_csv(
    os.path.join(
        EXP_DIR,
        "imputation_report.csv"
    ),
    index=False
)


# ============================================================


## 6. Feature Scaling (Training Data Only)

In [ ]:
# 6. SCALING - TRAINING DATA ONLY
# ============================================================

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

# Fit feature scaler using training observations only.
scaler_X.fit(
    processed.iloc[:train_end][FEATURE_COLS]
)

# Fit target scaler using observed training targets.
scaler_y.fit(
    processed.iloc[:train_end][["target_original"]]
    .dropna()
)

X_scaled = scaler_X.transform(
    processed[FEATURE_COLS]
).astype(np.float32)

y_scaled = np.full(
    n_total,
    np.nan,
    dtype=np.float32
)

valid_target = (
    processed["target_original"]
    .notna()
    .to_numpy()
)

y_scaled[valid_target] = (
    scaler_y.transform(
        processed.loc[
            valid_target,
            ["target_original"]
        ]
    ).ravel().astype(np.float32)
)

joblib.dump(
    scaler_X,
    os.path.join(
        EXP_DIR,
        "scaler_X.joblib"
    )
)

joblib.dump(
    scaler_y,
    os.path.join(
        EXP_DIR,
        "scaler_y.joblib"
    )
)

print("\n" + "=" * 70)
print("SCALING")
print("=" * 70)

print("Scaled X:", X_scaled.shape)
print("Scaled y:", y_scaled.shape)

print(
    "Missing scaled targets:",
    np.isnan(y_scaled).sum()
)

assert X_scaled.shape == (8016, 13)
assert y_scaled.shape == (8016,)
assert np.isfinite(X_scaled).all()
assert np.isnan(y_scaled).sum() == 206


# ============================================================


## 7. Generate LSTM Input Sequences

In [ ]:
# 7. GENERATE 24-HOUR INPUT SEQUENCES
# ============================================================

def make_sequences(
    X_data,
    y_data,
    timestamps,
    lookback,
    train_end,
    val_end
):

    containers = {
        split: {
            "X": [],
            "y": [],
            "time": []
        }
        for split in ["train", "val", "test"]
    }

    skipped_targets = 0

    for i in range(lookback, len(y_data)):

        # Exclude missing target values.
        if not np.isfinite(y_data[i]):
            skipped_targets += 1
            continue

        # Use only the previous 24 hourly observations.
        window = X_data[i-lookback:i]

        if not np.isfinite(window).all():
            continue

        # Assign based on target timestamp.
        if i < train_end:
            split = "train"

        elif i < val_end:
            split = "val"

        else:
            split = "test"

        containers[split]["X"].append(window)
        containers[split]["y"].append(y_data[i])
        containers[split]["time"].append(timestamps[i])

    result = {}

    for split in ["train", "val", "test"]:

        result[split] = {
            "X": np.asarray(
                containers[split]["X"],
                dtype=np.float32
            ).reshape(
                -1,
                lookback,
                X_data.shape[1]
            ),

            "y": np.asarray(
                containers[split]["y"],
                dtype=np.float32
            ).reshape(-1, 1),

            "time": pd.DatetimeIndex(
                containers[split]["time"]
            )
        }

    print(
        "\nSkipped missing targets:",
        skipped_targets
    )

    return result


data = make_sequences(
    X_scaled,
    y_scaled,
    processed["date_hour"].to_numpy(),
    LOOKBACK,
    train_end,
    val_end
)

X_train = data["train"]["X"]
y_train = data["train"]["y"]
time_train = data["train"]["time"]

X_val = data["val"]["X"]
y_val = data["val"]["y"]
time_val = data["val"]["time"]

X_test = data["test"]["X"]
y_test = data["test"]["y"]
time_test = data["test"]["time"]

print("\n" + "=" * 70)
print("EXPERIMENT L - SEQUENCE SHAPES")
print("=" * 70)

for name, X, y, timestamps in [
    ("Training", X_train, y_train, time_train),
    ("Validation", X_val, y_val, time_val),
    ("Testing", X_test, y_test, time_test)
]:

    print(
        f"{name}: X={X.shape}, y={y.shape}"
    )

    print(
        "Target dates:",
        timestamps.min(),
        "to",
        timestamps.max()
    )

assert X_train.shape == (5403, 24, 13)
assert X_val.shape == (1187, 24, 13)
assert X_test.shape == (1196, 24, 13)

assert np.isfinite(X_train).all()
assert np.isfinite(X_val).all()
assert np.isfinite(X_test).all()


# ============================================================


## 8. Build LSTM Model

In [ ]:
# 8. BUILD LSTM MODEL
# SAME ARCHITECTURE AS EXPERIMENT F
# ============================================================

model = Sequential([
    Input(shape=(LOOKBACK, N_FEATURES)),

    LSTM(
        64,
        return_sequences=True
    ),
    Dropout(0.2),

    LSTM(
        32,
        return_sequences=True
    ),
    Dropout(0.2),

    LSTM(16),
    Dropout(0.2),

    Dense(1)
])

model.compile(
    optimizer=Adam(
        learning_rate=LEARNING_RATE,
        clipnorm=1.0
    ),
    loss="mse",
    metrics=["mae"]
)

print("\n" + "=" * 70)
print("EXPERIMENT L - MODEL ARCHITECTURE")
print("=" * 70)

model.summary()

print(
    "\nTotal parameters:",
    model.count_params()
)


# ============================================================


## 9. Train LSTM Model

In [ ]:
# 9. TRAIN MODEL
# ============================================================

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=20,
        restore_best_weights=True,
        mode="min",
        verbose=1
    ),

    ModelCheckpoint(
        filepath=MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=10,
        min_lr=1e-6,
        mode="min",
        verbose=1
    )
]

history = model.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=False,
    callbacks=callbacks,
    verbose=1
)

history_df = pd.DataFrame(
    history.history
)

history_df.to_csv(
    os.path.join(
        EXP_DIR,
        "training_history.csv"
    ),
    index=False
)

best_epoch = (
    np.argmin(
        history.history["val_loss"]
    ) + 1
)

print("\n" + "=" * 70)
print("TRAINING SUMMARY")
print("=" * 70)

print("Epochs trained:", len(history_df))
print("Best epoch:", best_epoch)

print(
    "Best validation MSE (scaled):",
    float(
        np.min(
            history.history["val_loss"]
        )
    )
)


# ============================================================


## 10. Load Best Saved Model

In [ ]:
# 10. LOAD BEST SAVED MODEL
# ============================================================

best_model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)


# ============================================================


## 11. Define Evaluation Function

In [ ]:
# 11. EVALUATION FUNCTION
# ============================================================

def evaluate_split(
    model,
    X,
    y,
    timestamps,
    split_name
):

    pred_scaled = model.predict(
        X,
        verbose=0
    )

    pred = scaler_y.inverse_transform(
        pred_scaled.reshape(-1, 1)
    ).ravel()

    actual = scaler_y.inverse_transform(
        y.reshape(-1, 1)
    ).ravel()

    rmse = np.sqrt(
        mean_squared_error(actual, pred)
    )

    mae = mean_absolute_error(
        actual,
        pred
    )

    r2 = r2_score(
        actual,
        pred
    )

    predictions = pd.DataFrame({
        "date_hour": timestamps,
        "actual_kwh": actual,
        "predicted_kwh": pred
    })

    predictions["error_kwh"] = (
        predictions["predicted_kwh"]
        - predictions["actual_kwh"]
    )

    metrics = {
        "Dataset": split_name,
        "Samples": len(actual),
        "RMSE (kWh)": rmse,
        "MAE (kWh)": mae,
        "R² Score": r2
    }

    return metrics, predictions


# ============================================================


## 12. Evaluate Training, Validation and Testing

In [ ]:
# 12. EVALUATE TRAINING / VALIDATION / TESTING
# ============================================================

metrics_train, pred_train = evaluate_split(
    best_model,
    X_train,
    y_train,
    time_train,
    "Training"
)

metrics_val, pred_val = evaluate_split(
    best_model,
    X_val,
    y_val,
    time_val,
    "Validation"
)

metrics_test, pred_test = evaluate_split(
    best_model,
    X_test,
    y_test,
    time_test,
    "Testing"
)

metrics_df = pd.DataFrame([
    metrics_train,
    metrics_val,
    metrics_test
])

print("\n" + "=" * 70)
print("EXPERIMENT L - PERFORMANCE SUMMARY")
print("=" * 70)

print(
    metrics_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================


## 13. Save Evaluation Results

In [ ]:
# 13. SAVE RESULTS
# ============================================================

metrics_df.to_csv(
    os.path.join(
        EXP_DIR,
        "all_split_metrics.csv"
    ),
    index=False
)

for split_name, predictions in [
    ("training", pred_train),
    ("validation", pred_val),
    ("test", pred_test)
]:

    predictions.to_csv(
        os.path.join(
            EXP_DIR,
            f"{split_name}_predictions.csv"
        ),
        index=False
    )

print("\nPrediction and metric files saved.")


# ============================================================


## 14. Plot Training and Validation Loss

In [ ]:
# 14. PLOT TRAINING HISTORY
# ============================================================

plt.figure(figsize=(10, 5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("MSE (Scaled)")
plt.title("Experiment L - Training History")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(
        EXP_DIR,
        "training_loss.png"
    ),
    dpi=300
)

plt.show()


# ============================================================


## 15. Plot Actual vs Predicted Test Demand

In [ ]:
# 15. PLOT TEST PREDICTIONS
# ============================================================

plt.figure(figsize=(14, 5))

plt.plot(
    pred_test["date_hour"],
    pred_test["actual_kwh"],
    label="Actual",
    linewidth=1
)

plt.plot(
    pred_test["date_hour"],
    pred_test["predicted_kwh"],
    label="LSTM L - All Features",
    linewidth=1
)

plt.xlabel("Date")
plt.ylabel("Charging Demand (kWh)")
plt.title(
    "Experiment L - Actual vs Predicted Test Demand"
)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(
        EXP_DIR,
        "test_actual_vs_predicted.png"
    ),
    dpi=300
)

plt.show()


# ============================================================


## 16. Plot First 168 Test Observations

In [ ]:
# 16. PLOT FIRST 168 TEST OBSERVATIONS
# ============================================================

week_df = pred_test.head(168)

plt.figure(figsize=(14, 5))

plt.plot(
    week_df["date_hour"],
    week_df["actual_kwh"],
    label="Actual",
    linewidth=1.5
)

plt.plot(
    week_df["date_hour"],
    week_df["predicted_kwh"],
    label="LSTM L",
    linewidth=1.5
)

plt.xlabel("Date")
plt.ylabel("Charging Demand (kWh)")
plt.title(
    "Experiment L - First 168 Test Observations"
)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(
        EXP_DIR,
        "test_first_168_hours.png"
    ),
    dpi=300
)

plt.show()


# ============================================================


## 17. Matched-Timestamp Model Comparison

In [ ]:
# 17. MATCHED-TIMESTAMP COMPARISON
# EXPERIMENTS C, F, H, I, J, K AND L
# ============================================================

comparison_sources = {
    "C - Original All Features": os.path.join(
        ROOT,
        "Experiments",
        "LSTM_Experiment_C_2019",
        "test_predictions.csv"
    ),

    "F - Historical Only": os.path.join(
        ROOT,
        "Experiments",
        "LSTM_Experiment_F_Historical_Only_2019",
        "test_predictions.csv"
    ),

    "H - Historical + Traffic": os.path.join(
        ROOT,
        "Experiments",
        "LSTM_Experiment_H_Historical_Traffic_2019",
        "test_predictions.csv"
    ),

    "I - Historical + Weather": os.path.join(
        ROOT,
        "Experiments",
        "LSTM_Experiment_I_Historical_Weather_2019",
        "test_predictions.csv"
    ),

    "J - Historical + EV Sales": os.path.join(
        ROOT,
        "Experiments",
        "LSTM_Experiment_J_Historical_EV_Sales_2019",
        "test_predictions.csv"
    ),

    "K - Historical + Traffic + Weather": os.path.join(
        ROOT,
        "Experiments",
        "LSTM_Experiment_K_Historical_Traffic_Weather_2019",
        "test_predictions.csv"
    ),

    "L - All Features": os.path.join(
        EXP_DIR,
        "test_predictions.csv"
    )
}

frames = {}

for model_name, path in comparison_sources.items():

    if not os.path.exists(path):
        print("Skipping missing:", path)
        continue

    temp = pd.read_csv(
        path,
        parse_dates=["date_hour"]
    )

    temp.columns = [
        col.strip().lower()
        for col in temp.columns
    ]

    required = [
        "date_hour",
        "actual_kwh",
        "predicted_kwh"
    ]

    if not set(required).issubset(temp.columns):
        print("Skipping unsupported columns:", path)
        continue

    temp = temp[required].copy()

    if temp["date_hour"].duplicated().any():
        raise ValueError(
            f"Duplicate timestamps in {model_name}"
        )

    frames[model_name] = temp

if "L - All Features" not in frames:
    raise ValueError("Experiment L predictions not found.")

if len(frames) < 2:
    print(
        "No other completed experiment found. "
        "Skipping comparison."
    )

else:

    common_times = None

    for frame in frames.values():

        timestamps = set(
            frame["date_hour"]
        )

        common_times = (
            timestamps
            if common_times is None
            else common_times & timestamps
        )

    common_times = sorted(common_times)

    if not common_times:
        raise ValueError(
            "No common testing timestamps."
        )

    print("\n" + "=" * 70)
    print("MATCHED-TIMESTAMP COMPARISON")
    print("=" * 70)

    print("Models included:", list(frames.keys()))
    print("Common samples:", len(common_times))

    comparison_results = []
    reference_actual = None

    for model_name, frame in frames.items():

        matched = (
            frame[
                frame["date_hour"].isin(common_times)
            ]
            .sort_values("date_hour")
            .reset_index(drop=True)
        )

        actual = matched["actual_kwh"].to_numpy()
        predicted = matched["predicted_kwh"].to_numpy()

        if reference_actual is None:
            reference_actual = actual

        else:
            assert np.allclose(
                reference_actual,
                actual,
                rtol=1e-5,
                atol=1e-3
            ), (
                "Actual demand values differ "
                "between experiments."
            )

        comparison_results.append({
            "Model": model_name,
            "Samples": len(actual),

            "RMSE (kWh)": np.sqrt(
                mean_squared_error(
                    actual,
                    predicted
                )
            ),

            "MAE (kWh)": mean_absolute_error(
                actual,
                predicted
            ),

            "R² Score": r2_score(
                actual,
                predicted
            )
        })

    comparison_df = pd.DataFrame(
        comparison_results
    )

    print(
        comparison_df.to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )

    comparison_df.to_csv(
        os.path.join(
            EXP_DIR,
            "comparison_C_F_H_I_J_K_L_matched_metrics.csv"
        ),
        index=False
    )

    # ========================================================
    # 18. ALL-FEATURE CONTRIBUTION RELATIVE TO F
    # ========================================================

    if "F - Historical Only" in comparison_df["Model"].values:

        rmse_f = comparison_df.loc[
            comparison_df["Model"] ==
            "F - Historical Only",
            "RMSE (kWh)"
        ].iloc[0]

        rmse_l = comparison_df.loc[
            comparison_df["Model"] ==
            "L - All Features",
            "RMSE (kWh)"
        ].iloc[0]

        improvement = (
            (rmse_f - rmse_l) / rmse_f
        ) * 100

        print("\n" + "=" * 70)
        print("ALL-FEATURE CONTRIBUTION")
        print("=" * 70)

        print(
            f"RMSE improvement relative to F: "
            f"{improvement:.2f}%"
        )

        if improvement > 0:
            print(
                "Adding all external features reduced RMSE."
            )

        elif improvement < 0:
            print(
                "Adding all external features increased RMSE."
            )

        else:
            print(
                "No RMSE difference was observed."
            )


# ============================================================
# 19. SAVE EXPERIMENT MANIFEST
# ============================================================

manifest = {
    "experiment": "LSTM L",
    "dataset": "Dataset 4 - 2019",
    "features": FEATURE_COLS,
    "n_features": N_FEATURES,
    "lookback": LOOKBACK,
    "forecast_horizon": 1,
    "split": [0.70, 0.15, 0.15],
    "architecture": "LSTM 64-32-16",
    "dropout": 0.2,
    "loss": "mse",
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "max_epochs": EPOCHS,
    "seed": SEED,
    "test_samples": len(y_test),
    "best_epoch": int(best_epoch)
}

with open(
    os.path.join(
        EXP_DIR,
        "manifest.json"
    ),
    "w"
) as f:
    json.dump(
        manifest,
        f,
        indent=4
    )

print("\n" + "=" * 70)
print("EXPERIMENT L COMPLETED SUCCESSFULLY")
print("=" * 70)

print("Best model:", MODEL_PATH)
print("Results directory:", EXP_DIR)
